# PP-OCRv4 Mobile Rec 訓練（Colab GPU）

本筆記本用於在 Google Colab 上訓練 PP-OCRv4 mobile **文字辨識 (rec)** 模型。

**GitHub Repo:** `https://github.com/install88/trainingOcr`

**前置條件：** Det 模型已訓練完成；rec 資料集（crop 圖 + label txt）已準備並打包為 `rec_dataset.zip`。

**Rec 標注格式（每行）：**
```
crop_001.jpg\t2026.01.01
crop_002.jpg\tEXP:2026.01
```

**流程：**
1. Clone repo + 安裝環境
2. Mount Drive + 解壓 rec 資料集
3. 下載預訓練模型
4. Baseline 評估（訓練前）
5. 訓練 rec 模型（Block A 或 Block B）
6. 訓練後評估
7. 畫 Loss / Acc 曲線
8. 匯出推論模型 → 轉 ONNX

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. Clone Repo + 安裝環境

In [ ]:
import os
os.chdir("/content")

# Clone 專案 repo
if not os.path.exists("trainingOcr"):
    !git clone https://github.com/install88/trainingOcr.git

# Clone PaddleOCR 訓練框架
if not os.path.exists("PaddleOCR"):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

# 安裝 PaddlePaddle GPU 版（Colab 通常是 CUDA 12.x）
# 如果 Colab 用的是 CUDA 11.8，把 cu126 改成 cu118
!pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# 裝 PaddleOCR 官方 requirements（涵蓋 rapidfuzz / shapely / pyclipper / imgaug / lmdb 等）
!pip install -r /content/PaddleOCR/requirements.txt

# 匯出 ONNX 額外需要的
!pip install paddle2onnx onnxruntime

In [ ]:
# 驗證安裝
import paddle
print(f"PaddlePaddle: {paddle.__version__}")
print(f"GPU available: {paddle.is_compiled_with_cuda()}")
print(f"GPU count: {paddle.device.cuda.device_count()}")

In [ ]:
import os
os.chdir("/content")

if not os.path.exists("PaddleOCR"):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

os.chdir("/content/PaddleOCR")
!pwd

## 2. Mount Drive + 解壓資料集

把 `rec_dataset.zip` 放在 Google Drive 的 `My Drive/ocr_project/` 下。


In [ ]:
# 從 Google Drive 載入
from google.colab import drive
drive.mount('/content/drive')

# 假設資料集 zip 放在 Google Drive 的 My Drive/ocr_project/ 下
DRIVE_DATASET_PATH = "/content/drive/MyDrive/ocr_project/rec_dataset.zip"

!mkdir -p /content/dataset/rec
!unzip -o "{DRIVE_DATASET_PATH}" -d /content/dataset/rec/

## 3. 資料集結構

repo 已 clone 到 `/content/trainingOcr/`，但 crop 圖和標注不在 repo 裡（太大）。

你需要上傳 `rec_dataset.zip`（由本機工具產生的 `dataset/rec/` 打包）：
```
rec_dataset.zip
├── train/           ← 訓練用 crop 圖
├── val/             ← 驗證用 crop 圖
├── train_label.txt  ← 訓練標注（檔名\t文字）
└── val_label.txt    ← 驗證標注（val/檔名\t文字）
```

In [ ]:
# 檢查資料集
!echo "=== 資料集結構 ==="
!ls -la /content/dataset/rec/
!echo ""
!echo "=== 訓練集 crop 數量 ==="
!ls /content/dataset/rec/train/ | wc -l
!echo "=== 驗證集 crop 數量 ==="
!ls /content/dataset/rec/val/ | wc -l
!echo ""
!echo "=== train_label.txt 前 5 行 ==="
!head -5 /content/dataset/rec/train_label.txt
!echo ""
!echo "=== val_label.txt 前 5 行 ==="
!head -5 /content/dataset/rec/val_label.txt

## 4. 下載預訓練模型

In [ ]:
!mkdir -p /content/pretrained_models/PP-OCRv4_mobile_rec

# 下載 PP-OCRv4 mobile rec 預訓練權重
# PaddleX 發佈的 PP-OCRv4_mobile_rec_pretrained.pdparams（與 config 的
# SVTR_LCNet + MultiHead(CTC + NRTR) 架構相符）
!wget -nc -O /content/pretrained_models/PP-OCRv4_mobile_rec/best_accuracy.pdparams \
    https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv4_mobile_rec_pretrained.pdparams

print("預訓練模型下載完成！")
!ls -la /content/pretrained_models/PP-OCRv4_mobile_rec/

PRETRAIN = "/content/pretrained_models/PP-OCRv4_mobile_rec/best_accuracy"


## 5. 使用 repo 的訓練設定檔

直接用 repo 裡的 config，只需覆寫路徑為 Colab 環境的路徑。

In [ ]:
# 確認 repo config 存在
REPO_DIR = "/content/trainingOcr"
CONFIG_PATH = f"{REPO_DIR}/configs/rec/PP-OCRv4_mobile_rec_finetune.yml"
DICT_PATH = "/content/PaddleOCR/ppocr/utils/ppocr_keys_v1.txt"

!cat {CONFIG_PATH} | head -25
print(f"\n設定檔路徑：{CONFIG_PATH}")
print(f"字典路徑：{DICT_PATH}")
!wc -l {DICT_PATH}

## 6. Baseline 評估（訓練前）

先用 **官方 PP-OCRv4 rec 模型**（還沒 fine-tune）在我們自己的驗證集跑一次，記下 acc / norm_edit_dis 作為 baseline。

訓練完之後會再跑一次，直接比對數字看進步多少。


In [ ]:
os.chdir("/content/PaddleOCR")

# 用『還沒訓練過』的官方 PP-OCRv4 rec 模型在我們 val set 上評估
!python tools/eval.py \
    -c {CONFIG_PATH} \
    -o Global.use_gpu=true \
       Global.pretrained_model=/content/pretrained_models/PP-OCRv4_mobile_rec/best_accuracy \
       Global.character_dict_path={DICT_PATH} \
       Eval.dataset.data_dir=/content/dataset/rec/val/ \
       Eval.dataset.label_file_list=[/content/dataset/rec/val_label.txt]

print("\n=== BASELINE ===")
print("請記下上面的 acc / norm_edit_dis，這是『沒在我們資料上訓練過』的分數")


## 7. 開始訓練 — 下面有兩個區塊，**只能跑其中一個**

### 🅰️ 區塊 A：第一次訓練（或想砍掉重練）
- 從預訓練權重開始，epoch 從 0 數起
- 正常情況第一次跑這個

### 🅱️ 區塊 B：Colab 斷線後接續訓練
- 從 Drive 裡上次存的 `latest` checkpoint 接著跑
- Epoch 數、optimizer state、lr schedule **全部接續**，不會從頭開始
- **前提**：上面 `Clone Repo + 安裝環境`、`Mount Drive + 解壓 dataset`、`下載預訓練模型` 這三個區塊要先跑過

⚠️ **兩個只能跑一個**，跑錯會蓋掉進度
⚠️ 斷線後如果你不小心跑了 A，Drive 裡的 checkpoint 會被新訓練覆蓋，之前的進度就沒了

---

### ⬇️ 區塊 A：第一次訓練 ⬇️（下一格）

In [ ]:
os.chdir("/content/PaddleOCR")

# ⭐ checkpoint 直接存到 Google Drive：Colab 斷線也不會丟
# save_epoch_step 在 config 裡為 5，每 5 epoch 存一份 iter_epoch_N
# best_accuracy 每次 eval 產生新高時也會自動存（eval 頻率 eval_batch_step=500）
OUTPUT_DIR = "/content/drive/MyDrive/ocr_project/output_rec/"
LOG_PATH = "/content/drive/MyDrive/ocr_project/train_rec.log"
!mkdir -p "{OUTPUT_DIR}"
!mkdir -p /content/drive/MyDrive/ocr_project/

# 訓練時用 -o 覆寫 config 裡的路徑為 Colab 路徑
# config 已設定 epoch_num=100, learning_rate=0.0003, warmup_epoch=3
# 重跑訓練會覆蓋 train_rec.log，要保留舊的請先改檔名
!python tools/train.py \
    -c {CONFIG_PATH} \
    -o Global.use_gpu=true \
       Global.pretrained_model=/content/pretrained_models/PP-OCRv4_mobile_rec/best_accuracy \
       Global.save_model_dir="{OUTPUT_DIR}" \
       Global.save_inference_dir="{OUTPUT_DIR}inference/" \
       Global.character_dict_path={DICT_PATH} \
       Train.dataset.data_dir=/content/dataset/rec/train/ \
       Train.dataset.label_file_list=[/content/dataset/rec/train_label.txt] \
       Eval.dataset.data_dir=/content/dataset/rec/val/ \
       Eval.dataset.label_file_list=[/content/dataset/rec/val_label.txt] \
    2>&1 | tee "{LOG_PATH}"


---

### ⬇️ 區塊 B：斷線後接續訓練 ⬇️（下一格）

**什麼時候跑這格：** Colab 斷線、關掉、當機、運算單元用完重開後，想接著上次 epoch 繼續跑

**跑之前先做：**
1. 重開 Colab session
2. 跑過 `Clone Repo + 安裝環境`
3. 跑過 `Mount Drive + 解壓 dataset`
4. 跑過 `下載預訓練模型`（resume 用不到但 config 驗證需要）
5. **跳過**「區塊 A：第一次訓練」那一格 ← 最重要
6. 跑下面這一格

跑完你會看到：`resume from /content/drive/.../latest, current epoch is N`

In [ ]:
import os
os.chdir("/content/PaddleOCR")

# === Drive 上的 checkpoint 路徑 ===
CKPT_DIR = "/content/drive/MyDrive/ocr_project/output_rec"
LATEST_PDPARAMS = f"{CKPT_DIR}/latest.pdparams"
LATEST_PDOPT = f"{CKPT_DIR}/latest.pdopt"

# === 先檢查 Drive 裡有沒有 checkpoint ===
print("=== Drive 裡的 checkpoint 檔案 ===")
!ls -la "{CKPT_DIR}" 2>/dev/null || echo "❌ 找不到目錄 {CKPT_DIR}（Drive 沒 mount？還是上次沒存成功？）"
print()

if not os.path.exists(LATEST_PDPARAMS):
    print(f"❌ 找不到 {LATEST_PDPARAMS}")
    print("   → 不能 resume，請改跑『區塊 A：第一次訓練』")
    print("   可能原因：")
    print("   1. Drive 沒 mount（先跑 Mount Drive 那格）")
    print("   2. 上次訓練沒跑到 save_epoch_step=5 就斷了（連一個 checkpoint 都沒存）")
    print("   3. 路徑打錯 / Drive 資料夾被手動刪了")
else:
    size_mb = os.path.getsize(LATEST_PDPARAMS) / 1024 / 1024
    print(f"✅ 找到 latest.pdparams（{size_mb:.1f} MB），可以接續訓練")
    print()
    print("=== 開始接續訓練 ===")

    # Global.checkpoints=（注意不是 pretrained_model）→ 連 optimizer state 和 epoch 數一起恢復
    # tee -a 是 append 模式，log 會接在上次後面，不會覆蓋
    !python tools/train.py \
        -c {CONFIG_PATH} \
        -o Global.use_gpu=true \
           Global.checkpoints={CKPT_DIR}/latest \
           Global.save_model_dir={CKPT_DIR}/ \
           Global.save_inference_dir={CKPT_DIR}/inference/ \
           Global.character_dict_path={DICT_PATH} \
           Train.dataset.data_dir=/content/dataset/rec/train/ \
           Train.dataset.label_file_list=[/content/dataset/rec/train_label.txt] \
           Eval.dataset.data_dir=/content/dataset/rec/val/ \
           Eval.dataset.label_file_list=[/content/dataset/rec/val_label.txt] \
        2>&1 | tee -a /content/drive/MyDrive/ocr_project/train_rec.log


In [ ]:
# ============================================================
# AFTER TRAINING: 用 fine-tune 後的 best_accuracy 在 val 上再跑一次
# 和 baseline 直接比對 acc / norm_edit_dis，就知道進步多少
# checkpoint 在 Drive，路徑指過去
# ============================================================
!python tools/eval.py \
    -c {CONFIG_PATH} \
    -o Global.use_gpu=true \
       Global.pretrained_model=/content/drive/MyDrive/ocr_project/output_rec/best_accuracy \
       Global.character_dict_path={DICT_PATH} \
       Eval.dataset.data_dir=/content/dataset/rec/val/ \
       Eval.dataset.label_file_list=[/content/dataset/rec/val_label.txt]

print("\n=== AFTER TRAINING ===")
print("和上面 baseline 的 acc 比一下，差距就是這次 fine-tune 的效果")


## 8. 畫 Loss / Eval 曲線

解析 `train_rec.log` 畫訓練 loss 和驗證 acc / norm_edit_dis 曲線。
即使 Colab 斷線只跑到一半也能畫，跑到哪畫到哪。

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

LOG_PATH = "/content/drive/MyDrive/ocr_project/train_rec.log"

with open(LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

# 解析每個 training step 的 loss
loss_re = re.compile(r"epoch:\s*\[(\d+)/\d+\].*?global_step:\s*(\d+).*?loss:\s*([\d.]+)")
# 解析每次 eval 的 metric（rec：acc + norm_edit_dis）
eval_re = re.compile(r"cur metric.*?acc:\s*([\d.]+).*?norm_edit_dis:\s*([\d.]+)")

epochs, steps, losses = [], [], []
eval_steps, accs, neds = [], [], []
cur_step = 0

for line in text.splitlines():
    m = loss_re.search(line)
    if m:
        epochs.append(int(m.group(1)))
        cur_step = int(m.group(2))
        steps.append(cur_step)
        losses.append(float(m.group(3)))
        continue
    m = eval_re.search(line)
    if m:
        eval_steps.append(cur_step)
        accs.append(float(m.group(1)))
        neds.append(float(m.group(2)))

print(f"解析到 {len(losses)} 筆 training loss、{len(accs)} 筆 eval metric")
if epochs:
    print(f"目前跑到 epoch {epochs[-1]} / global_step {steps[-1]}")
if accs:
    best_i = int(np.argmax(accs))
    print(f"目前最佳 acc = {accs[best_i]:.4f} (在 step {eval_steps[best_i]})")

if not losses:
    print("⚠️ 還沒解析到任何 loss。確認 LOG_PATH 檔案是否存在，或訓練是否已開始。")
else:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    ax[0].plot(steps, losses, alpha=0.3, label="raw")
    if len(losses) >= 20:
        w = max(20, len(losses) // 50)
        smooth = np.convolve(losses, np.ones(w) / w, mode="valid")
        ax[0].plot(steps[w - 1:], smooth, linewidth=2, label=f"smoothed (w={w})")
    ax[0].set_xlabel("global_step")
    ax[0].set_ylabel("loss")
    ax[0].set_title(f"Training Loss  (up to epoch {epochs[-1]})")
    ax[0].legend(); ax[0].grid(alpha=0.3)

    if accs:
        ax[1].plot(eval_steps, accs, "o-", linewidth=2, label="acc")
        ax[1].plot(eval_steps, neds, "s--", alpha=0.6, label="norm_edit_dis")
        ax[1].set_ylim(0, 1)
        ax[1].set_title(f"Eval Metrics  (best acc={max(accs):.4f})")
    else:
        ax[1].text(0.5, 0.5, "尚無 eval 資料\n(至少要跑滿一個 eval interval)",
                   ha="center", va="center", transform=ax[1].transAxes)
        ax[1].set_title("Eval Metrics")
    ax[1].set_xlabel("global_step"); ax[1].legend(); ax[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/ocr_project/train_rec_curves.png", dpi=120)
    plt.show()
    print("\n✅ 圖已存到 /content/drive/MyDrive/ocr_project/train_rec_curves.png")


## 9. 匯出推論模型

In [ ]:
!python tools/export_model.py \
    -c {CONFIG_PATH} \
    -o Global.pretrained_model=/content/drive/MyDrive/ocr_project/output_rec/best_accuracy \
       Global.character_dict_path={DICT_PATH} \
       Global.save_inference_dir=/content/drive/MyDrive/ocr_project/output_rec/inference/

print("推論模型已匯出到 Drive！")
!ls -la /content/drive/MyDrive/ocr_project/output_rec/inference/


## 10. 轉換為 ONNX

In [ ]:
!mkdir -p /content/drive/MyDrive/ocr_project/output_rec/onnx

!python -m paddle2onnx.convert \
    --model_dir /content/drive/MyDrive/ocr_project/output_rec/inference/ \
    --model_filename inference.pdmodel \
    --params_filename inference.pdiparams \
    --save_file /content/drive/MyDrive/ocr_project/output_rec/onnx/ch_PP-OCRv4_rec_infer.onnx \
    --opset_version 11 \
    --enable_onnx_checker True

print("ONNX 轉換完成！")
!ls -la /content/drive/MyDrive/ocr_project/output_rec/onnx/

In [ ]:
# 驗證 ONNX 模型
import onnxruntime as ort
session = ort.InferenceSession("/content/drive/MyDrive/ocr_project/output_rec/onnx/ch_PP-OCRv4_rec_infer.onnx")
for inp in session.get_inputs():
    print(f"Input:  {inp.name} shape={inp.shape} type={inp.type}")
for out in session.get_outputs():
    print(f"Output: {out.name} shape={out.shape} type={out.type}")
print("\nONNX 模型驗證通過！")